# Solving the Steady-State 2D RTE using a PINN on the Integral (Peierls) Form

In this notebook, we solve the steady-state 2D Radiative Transfer Equation (RTE) in a participating square medium by training a network **directly on the smooth incident radiation $G(x, y)$**, using the **integral form** of the RTE (the formulation solved by Crosbie & Schrenker), instead of the 4D intensity $I(x, y, \mu, \eta)$.

### Case 2 (2D)
Same configuration as the strong-form and VPINN notebooks:
* **Geometry:** $0 \le x \le L_x$ and $0 \le y \le L_y$ with $L_x = L_y = 1.0\text{ m}$ (Square medium).
* **Physical Properties:** $\sigma = 1.0\text{ m}^{-1}$ (isotropic scattering), $\kappa = 0\text{ m}^{-1}$.
* **Boundary Conditions:** diffuse radiation $I = 1$ entering through the top wall ($y=1$); the three other walls are cold ($I = 0$ incoming).

### What changes compared to the $I$-based PINNs
Integrating the RTE along the rays and over all directions turns it into an integral equation for $G$ alone:

$$G(\vec r) = \underbrace{G_0(\vec r)}_{\text{uncollided (hot wall)}} + \int_{4\pi}\!\!\int_0^{p_b} e^{-\sigma p/\sin\theta}\,\frac{\sigma}{4\pi}\,G(\vec r - p\,\hat d)\,\frac{dp}{\sin\theta}\,d\Omega$$

The network is $G_\theta(x, y)$ (2 inputs) and the residual is $R = G_\theta - G_0 - \mathcal K[G_\theta]$. The operator $\mathcal K$ is a **precomputed backward-ray integration** (geometry computed once; only $G_\theta$ at the ray points is re-evaluated during training). No autograd in the loss. Training is **mini-batched** on a wall-clock budget so it fits in 8 GB of GPU memory.

**Why this helps:** the angular discontinuity of $I$ (the hard part for a smooth network — the honest comparison: the strong-form $I$-PINN reaches ~5–15%, and the not-yet-converged upwind VPINN run stalled at ~40%) disappears entirely: $G$ is a smooth 2D function. The difficulty moves into the deterministic transport operator $\mathcal K$ — a much better trade.

**Caveats:** the accuracy floor is set by the operator resolution (`N_THETA` × `N_PHI` directions, `N_T` ray steps — the operator alone reproduces the DOM to ~1.5%) and by `N_COL`; all are memory-bound knobs. Validation as always: the in-notebook DOM reference and the exact anchor $G(0.5, 0.5) = \pi$.

In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---------------- KNOBS ----------------
SIGMA          = 1.0
N_COL          = 3000     # collocation points
N_THETA        = 10       # polar quadrature (ray directions)
N_PHI          = 32       # azimuthal quadrature   -> Q = 320 directions
N_T            = 48       # samples along each ray
BATCH          = 64      # collocation points per step (mini-batch -> fits 8 GB, peak ~3.6 GB)
TIME_BUDGET_MIN = 25      # wall-clock training budget
# If OOM on your GPU: lower BATCH (then N_PHI / N_T). More directions/steps -> lower error, more memory.

device: cuda


## 1. Network Architecture
A small MLP for the smooth scalar field $G(x, y)$ — only **2 inputs**, no angular variables.

In [2]:
class GNet(nn.Module):
    def __init__(self, h=64, nl=4):
        super().__init__()
        L = [nn.Linear(2, h), nn.Tanh()]
        for _ in range(nl - 1):
            L += [nn.Linear(h, h), nn.Tanh()]
        L += [nn.Linear(h, 1)]
        self.net = nn.Sequential(*L)
    def forward(self, x):
        return self.net(x)

net = GNet().to(device)

## 2. Reference: 2D DOM Solver + Exact Anchor
Same solver as in the other Case 2 notebooks (implicit upwind sweeps + source iteration). Reminder of the exact anchor: for a purely scattering square with a single hot wall, $G(0.5, 0.5) = \pi$ exactly.

In [ ]:
# Solveur DOM 2D de reference : meme equation et memes BCs que le PINN.
# Schema upwind implicite, balayages vectorises par fronts d'onde (les cellules
# d'une anti-diagonale ne dependent que de la precedente), iteration de la source.

def resoudre_dom(M=201, N_xi=12, N_phi=48, beta=1.0, sigma_dom=1.0, tol=1e-8, max_iter=2000):
    dx = 1.0 / (M - 1)

    nx, wx = np.polynomial.legendre.leggauss(N_xi)
    xi = 0.5 * (nx + 1.0)
    phi = (np.arange(N_phi) + 0.5) * 2.0 * np.pi / N_phi
    XI, PHI = np.meshgrid(xi, phi, indexing='ij')
    W = np.outer(wx, np.full(N_phi, 2.0 * np.pi / N_phi)).ravel()
    st = np.sqrt(1.0 - XI**2)
    MU = (st * np.cos(PHI)).ravel()
    ETA = (st * np.sin(PHI)).ravel()

    quadrants = []
    for smu in (1, -1):
        for seta in (1, -1):
            sel = (np.sign(MU) == smu) & (np.sign(ETA) == seta)
            quadrants.append((smu, seta, MU[sel], ETA[sel], W[sel]))

    def balayage(smu, seta, mu, eta, S):
        a = np.abs(mu)[:, None] / dx
        b = np.abs(eta)[:, None] / dx
        I = np.zeros((mu.size, M, M))
        bc_y = 1.0 if seta < 0 else 0.0   # paroi haute (y=1) chaude, les autres a 0
        Sf = S[::smu, ::seta]             # vues retournees : balayage toujours croissant
        If = I[:, ::smu, ::seta]
        denom = a + b + beta
        for d in range(2 * M - 1):
            ii = np.arange(max(0, d - M + 1), min(d, M - 1) + 1)
            jj = d - ii
            Ix = If[:, np.where(ii > 0, ii - 1, 0), jj]
            Ix[:, ii == 0] = 0.0
            Iy = If[:, ii, np.where(jj > 0, jj - 1, 0)]
            Iy[:, jj == 0] = bc_y
            If[:, ii, jj] = (a * Ix + b * Iy + Sf[ii, jj][None, :]) / denom
        return I

    G = np.zeros((M, M))
    for it in range(max_iter):
        S = (sigma_dom / (4.0 * np.pi)) * G
        Gn = np.zeros_like(G)
        for smu, seta, mu, eta, w in quadrants:
            Gn += np.tensordot(w, balayage(smu, seta, mu, eta, S), axes=(0, 0))
        diff = np.max(np.abs(Gn - G))
        G = Gn
        if diff < tol:
            break
    print(f"DOM converge en {it} iterations (diff = {diff:.2e})")
    return G

M_DOM = 201
G_dom = resoudre_dom(M=M_DOM, beta=SIGMA, sigma_dom=SIGMA)
grid = np.linspace(0.0, 1.0, M_DOM)
mid = (M_DOM - 1) // 2
print(f"G(0.5, 0.5) DOM = {G_dom[mid, mid]:.4f}   (exact = pi = {np.pi:.4f})")

## 3. Precomputation of the Integral Operator $\mathcal K$
For each collocation point and each of the $Q$ ray directions: backward ray to the boundary, uncollided contribution $G_0$ if the ray exits through the hot wall, and the attenuated in-scattering coefficients along the ray (trapezoidal rule, `N_T` steps). Geometry and coefficients are computed **once**; training only re-evaluates $G_\theta$ at the stored ray points.

In [4]:
# ---------------- precompute the integral operator (geometry only) ----------------
nx, wx = np.polynomial.legendre.leggauss(N_THETA)
xi = 0.5 * (nx + 1.0)                                # xi in [0,1] (weight not halved -> both hemispheres)
phis = (np.arange(N_PHI) + 0.5) * 2*np.pi / N_PHI
DIRS = []
for i in range(N_THETA):
    st = np.sqrt(1 - xi[i]**2)
    for ph in phis:
        DIRS.append((st*np.cos(ph), st*np.sin(ph), st, wx[i]*(2*np.pi/N_PHI)))
DIRS = np.array(DIRS); Q = len(DIRS)                 # sum(w) = 4*pi
print(f"{Q} ray directions")

def dist_to_boundary(rx, ry, dx, dy):
    c = []
    if dx >  1e-12: c.append((rx/dx, 2))
    if dx < -1e-12: c.append(((rx-1)/dx, 3))
    if dy >  1e-12: c.append((ry/dy, 0))
    if dy < -1e-12: c.append(((ry-1)/dy, 1))
    c = [(p, w) for p, w in c if p > 1e-9]
    return min(c) if c else (0.0, -1)

cols = np.random.rand(N_COL, 2) * 0.98 + 0.01
QT = Q * N_T
G0 = np.zeros(N_COL)
RP = np.zeros((N_COL, QT, 2), np.float32)
CF = np.zeros((N_COL, QT), np.float32)
t0 = time.time()
for i in range(N_COL):
    rx, ry = cols[i]
    for q, (mu, eta, st, w) in enumerate(DIRS):
        dxp, dyp = mu/st, eta/st
        pb, wall = dist_to_boundary(rx, ry, dxp, dyp)
        sl = slice(q*N_T, (q+1)*N_T)
        if pb <= 0:
            RP[i, sl, 0] = rx; RP[i, sl, 1] = ry; continue
        if wall == 1:
            G0[i] += w * np.exp(-SIGMA * pb / st)
        ps = np.linspace(0, pb, N_T); dp = ps[1] - ps[0]
        tw = np.full(N_T, dp); tw[0] *= 0.5; tw[-1] *= 0.5
        RP[i, sl, 0] = rx - ps*dxp; RP[i, sl, 1] = ry - ps*dyp
        CF[i, sl] = w * np.exp(-SIGMA*ps/st) * (SIGMA/(4*np.pi))/st * tw
print(f"precompute: {time.time()-t0:.0f}s  (RP storage {RP.nbytes/1e6:.0f} MB)")

RP_t   = torch.tensor(RP, device=device)
CF_t   = torch.tensor(CF, device=device)
G0_t   = torch.tensor(G0, dtype=torch.float32, device=device).view(-1, 1)
cols_t = torch.tensor(cols, dtype=torch.float32, device=device)

320 ray directions
precompute: 40s  (RP storage 369 MB)


## 4. Model Training (mini-batched, wall-clock budget)
Residual $R = G_\theta - G_0 - \mathcal K[G_\theta]$ minimized with Adam on random mini-batches of collocation points (fits in 8 GB), with step-wise lr decay, on a fixed wall-clock budget. The error vs DOM is logged every minute.

In [5]:
# ---------------- G-error vs DOM ----------------
XX, YY = np.meshgrid(grid, grid, indexing='ij')
grid_t = torch.tensor(np.stack([XX.ravel(), YY.ravel()], 1), dtype=torch.float32, device=device)
mask = grid >= 0.05
def gerr():
    with torch.no_grad():
        Gp = net(grid_t).cpu().numpy().reshape(M_DOM, M_DOM)
    return np.linalg.norm((Gp - G_dom)[np.ix_(mask, mask)]) / np.linalg.norm(G_dom[np.ix_(mask, mask)])

# ---------------- mini-batched training on a wall-clock budget ----------------
opt = optim.Adam(net.parameters(), lr=2e-3)
budget = TIME_BUDGET_MIN * 60.0
hist_t, hist_err = [], []
t0 = time.time(); step = 0; next_log = 0.0
while time.time() - t0 < budget:
    frac = (time.time() - t0) / budget
    for g in opt.param_groups:                       # lr decay 2e-3 -> ~2.5e-4
        g['lr'] = 2e-3 * (0.5 ** int(frac / 0.25))
    idx = torch.randint(0, N_COL, (BATCH,), device=device)
    opt.zero_grad()
    vals = net(RP_t[idx].reshape(-1, 2)).view(BATCH, QT)
    Kg = (CF_t[idx] * vals).sum(1, keepdim=True)
    loss = torch.mean((net(cols_t[idx]) - G0_t[idx] - Kg) ** 2)
    loss.backward(); opt.step(); step += 1
    if time.time() - t0 >= next_log:
        e = gerr(); hist_t.append((time.time()-t0)/60); hist_err.append(e)
        print(f"t {(time.time()-t0)/60:4.1f} min | step {step:6d} | loss {loss.item():.2e} | G-err vs DOM {e*100:.2f}%")
        next_log += 60.0
final = gerr()
print(f"\nFINISHED: {step} steps in {(time.time()-t0)/60:.1f} min | FINAL G-err vs DOM = {final*100:.2f}%")

t  0.0 min | step      1 | loss 5.50e+00 | G-err vs DOM 98.68%
t  1.0 min | step   1872 | loss 6.64e-03 | G-err vs DOM 2.25%
t  2.0 min | step   3745 | loss 3.27e-03 | G-err vs DOM 2.69%
t  3.0 min | step   5631 | loss 7.74e-03 | G-err vs DOM 1.58%
t  4.0 min | step   7513 | loss 5.91e-03 | G-err vs DOM 1.58%
t  5.0 min | step   9400 | loss 8.01e-03 | G-err vs DOM 3.21%
t  6.0 min | step  11276 | loss 6.57e-03 | G-err vs DOM 2.35%
t  7.0 min | step  13152 | loss 5.26e-03 | G-err vs DOM 1.24%
t  8.0 min | step  15002 | loss 4.37e-03 | G-err vs DOM 1.71%
t  9.0 min | step  16808 | loss 4.36e-03 | G-err vs DOM 1.63%
t 10.0 min | step  18640 | loss 3.13e-03 | G-err vs DOM 1.74%
t 11.0 min | step  20491 | loss 3.22e-03 | G-err vs DOM 2.12%
t 12.0 min | step  22355 | loss 3.30e-03 | G-err vs DOM 2.51%
t 13.0 min | step  24239 | loss 1.87e-03 | G-err vs DOM 2.27%
t 14.0 min | step  26125 | loss 1.96e-03 | G-err vs DOM 2.33%
t 15.0 min | step  28010 | loss 2.80e-03 | G-err vs DOM 1.99%
t 16.0 

## 5. Results: $G(x, y)$ and Centerline Comparison vs DOM

In [ ]:
# ---------------- results ----------------
with torch.no_grad():
    G_pred = net(grid_t).cpu().numpy().reshape(M_DOM, M_DOM)

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
im = ax[0].pcolormesh(XX, YY, G_pred, cmap='jet', shading='auto')
ax[0].set_title(f"G(x,y) radial-PINN (err {final*100:.1f}%)"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
fig.colorbar(im, ax=ax[0])

ax[1].plot(grid, G_pred[mid, :], 'r-', lw=2, label="radial-PINN")
ax[1].plot(grid, G_dom[mid, :], 'k--', lw=1.5, label="DOM")
ax[1].plot(0.5, np.pi, 'b*', ms=12, label=r"exact $G(0.5,0.5)=\pi$")
ax[1].set_xlabel("y"); ax[1].set_ylabel("G(0.5, y)"); ax[1].set_title("Vertical centerline"); ax[1].legend(); ax[1].grid(alpha=.3)

ax[2].plot(grid, G_pred[:, mid], 'b-', lw=2, label="radial-PINN")
ax[2].plot(grid, G_dom[:, mid], 'k--', lw=1.5, label="DOM")
ax[2].plot(0.5, np.pi, 'b*', ms=12)
ax[2].set_xlabel("x"); ax[2].set_ylabel("G(x, 0.5)"); ax[2].set_title("Horizontal centerline"); ax[2].legend(); ax[2].grid(alpha=.3)

plt.tight_layout(); plt.savefig("radial_pinn_cas2.png", dpi=140); plt.show()

plt.figure(figsize=(6, 4))
plt.plot(hist_t, np.array(hist_err)*100, 'o-')
plt.xlabel("time (min)"); plt.ylabel("G-err vs DOM (%)"); plt.title("Convergence"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("radial_pinn_cas2_conv.png", dpi=140); plt.show()

# Erreur relative PINN vs DOM sur la ligne verticale centrale (meme format que le cas 2)
ref_y = np.linspace(0.0, 1.0, 11)
ref_G_vertical = np.interp(ref_y, grid, G_dom[mid, :])
G_pinn_at_ref = np.interp(ref_y, grid, G_pred[mid, :])
err_v = np.abs(G_pinn_at_ref - ref_G_vertical) / np.abs(ref_G_vertical)
print("y            :", np.round(ref_y, 2))
print("G DOM        :", np.round(ref_G_vertical, 3))
print("G PINN       :", np.round(G_pinn_at_ref, 3))
print("erreur rel.  :", np.round(100 * err_v, 1), "%")

## 6. Conclusion

- Solving the **integral (Peierls) form for $G(x, y)$** removes the angular discontinuity that penalizes the $I$-based approaches: the network's unknown is a smooth 2D function, and this mesh-free run reaches **~2% vs the DOM** in 25 minutes (centre $\to \pi$). Honest comparison: strong-form $I$-PINN ~5–15%; upwind VPINN ~40% *before convergence* (L-BFGS stopped mid-descent — to be re-run).
- The hard part became **building the transport operator** $\mathcal K$ (deterministic, precomputed backward-ray integration) — a much better trade than fighting a discontinuity with a smooth network.
- Accuracy is bounded by the operator resolution (`N_THETA`, `N_PHI`, `N_T`) and `N_COL` — all memory-bound; mini-batching lets us push them while fitting 8 GB.
- Same philosophy as the **Crosbie–Schrenker integral equation**, but mesh-free and differentiable: the kernel coefficients $e^{-\sigma p/\sin\theta}\,(\sigma/4\pi)/\sin\theta$ depend explicitly on $\sigma$, so keeping $(p, \sin\theta)$ precomputed and rebuilding the coefficients inside the torch graph makes the whole model **differentiable w.r.t. $\sigma$** — the natural route to the **inverse problem**.